In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
from scipy import stats

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

engine = connect_to_db()

conn = engine.connect()

df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

In [ ]:
features = [
    "is_mobile",
    "cnt",
    "advance_booking_days",
    "is_distance_unknown",
    "trip_type",
    "channel"
]

df_model = df[features + ["is_booking"]].copy()

df_model = pd.get_dummies(df_model, drop_first=True)

df_model = df_model.dropna()

X = df_model.drop("is_booking", axis=1)
y = df_model["is_booking"]

X = X.astype(float)

X = sm.add_constant(X)

model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

In [ ]:
conn.close()

## Logistic Regression Insights

A logistic regression model was used to control for key behavioral and contextual variables.

The results confirm that mobile usage has a significant negative impact on booking probability:

- Mobile coefficient: -0.277 (p < 0.001)
- Odds ratio: ~0.76

This indicates that mobile users are approximately 24% less likely to complete a booking compared to desktop users, even after controlling for session intensity, booking window, distance availability, and trip characteristics.

These findings are consistent with the observational A/B test, which showed a ~27% lower conversion rate for mobile users.

## Additional Insights

- Higher session intensity (`cnt`) is associated with lower conversion, suggesting exploratory behavior.
- Longer booking windows reduce conversion likelihood.
- Missing distance information negatively impacts booking probability.
- Solo travelers show higher conversion rates compared to groups.

## Conclusion

The negative effect of mobile on conversion appears to be robust and not explained by observable behavioral variables, suggesting structural differences in user experience or context across devices.